In [62]:
import os 
import time
import random
from pathlib import Path
from typing import  Dict, List, Optional
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup
from cryptography.fernet import Fernet

base_url = "https://books.toscrape.com/"
output_dir = Path("scraped_books")
output_dir.mkdir(exist_ok=True)
ENC_PATH = output_dir / "books_scraped.csv.enc"


In [63]:

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/116.0 Safari/537.36 "
        "(Educational scraper for learning; contact: manishk152003@gmail.com)"
    )
}


In [64]:

REQUEST_TIMEOUT = 15  # seconds
RETRY_COUNT = 3
DELAY_RANGE = (1.0, 2.0)  # seconds between requests (ethical rate limiting)
MAX_PAGES = 1  # scrape first N pages (you can increase)


In [66]:

def get_request(session: requests.Session, url: str) -> Optional[requests.Response]:
    for attempt in range(1, RETRY_COUNT + 1):
        try:
            resp = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            resp.encoding = 'utf-8'
            if 200 <= resp.status_code < 300:
                time.sleep(random.uniform(*DELAY_RANGE))
                return resp
            else:
                print(f"[WARN] {url} -> HTTP {resp.status_code}")
        except requests.RequestException as e:
            print(f"[ERROR] Attempt {attempt}/{RETRY_COUNT} for {url}: {e}")
        # Backoff before retry
        time.sleep(0.5 * attempt)
    return None


In [67]:
session = requests.Session()
get_request(session,base_url)

<Response [200]>

In [68]:

def parse_rating_from_class(class_list: List[str]) -> Optional[int]:
    mapping = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    for c in class_list:
        if c in mapping:
            return mapping[c]
    return None


In [69]:

def parse_list_page(html: str, page_url: str) -> List[Dict]:
    soup = BeautifulSoup(html, "html.parser")
    items = []
    for card in soup.select("article.product_pod"):
        a = card.select_one("h3 a")
        title = a.get("title", "").strip()
        relative_href = a.get("href", "")
        product_url = urljoin(page_url, relative_href)

        price_el = card.select_one(".price_color")
        price_text = price_el.text.strip() if price_el else ""

        rating_el = card.select_one(".star-rating")
        rating = parse_rating_from_class(rating_el.get("class", [])) if rating_el else None

        availability_el = card.select_one(".availability")
        availability = availability_el.text.strip() if availability_el else ""

        items.append(
            {
                "title": title,
                "price": price_text,  # e.g. '£51.77'
                "rating": rating,     # 1-5
                "availability": availability,
                "product_page_url": product_url,
            }
        )
    return items



In [70]:

def parse_detail_page(html: str) -> Dict[str, Optional[str]]:
    soup = BeautifulSoup(html, "html.parser")
    upc = None
    for row in soup.select("table.table.table-striped tr"):
        th = row.find("th")
        td = row.find("td")
        if th and td and th.text.strip().lower() == "upc":
            upc = td.text.strip()
    category = None
    bc = soup.select("ul.breadcrumb li a")
    if len(bc) >= 3:
        category = bc[2].text.strip()

    desc_el = soup.select_one("#product_description ~ p")
    description = desc_el.text.strip() if desc_el else None

    return {
        "upc": upc,
        "category": category,
        "description": description,
    }


In [71]:

def scrape_books(max_pages: int = MAX_PAGES) -> pd.DataFrame:
    session = requests.Session()
    all_rows: List[Dict] = []

    first_page_url = urljoin(base_url, "index.html")
    page_urls = [first_page_url]
   
    for i in range(2, max_pages + 1):
        page_urls.append(urljoin(base_url, f"catalogue/page-{i}.html"))

    for page_url in page_urls:
        print(f"[INFO] Fetching list page: {page_url}")
        resp = get_request(session, page_url)
        if resp is None:
            print(f"[ERROR] Skipping page due to repeated failures: {page_url}")
            continue

        shallow_items = parse_list_page(resp.text, page_url)
        print(f"[INFO] Found {len(shallow_items)} items on this page.")

        for item in shallow_items:
            detail_url = item["product_page_url"]
            print(f"  [INFO] Fetching detail: {detail_url}")
            dresp = get_request(session, detail_url)
            if dresp is None:
                print(f"  [WARN] Skipping detail due to error: {detail_url}")
                all_rows.append({**item, "upc": None, "category": None, "description": None})
                continue
            detail = parse_detail_page(dresp.text)
            all_rows.append({**item, **detail})

    df = pd.DataFrame(all_rows)
    if not df.empty:
        df["price_numeric"] = (
            df["price"]
            .str.replace("£", "", regex=False)
            .str.replace(",", "", regex=False)
            .astype(float)
        )
    return df


In [72]:

def get_fernet() -> Fernet:
    """Get a Fernet instance from env var FERNET_KEY or generate & persist one."""
    key = os.getenv("FERNET_KEY")
    key_file = output_dir / ".fernet.key"

    if key:
        return Fernet(key.encode())

    if key_file.exists():
        key = key_file.read_text().strip()
        return Fernet(key.encode())

    # Generate a new key and persist it locally (chmod 600)
    new_key = Fernet.generate_key()
    key_file.write_text(new_key.decode(), encoding="utf-8")
    try:
        os.chmod(key_file, 0o600)
    except Exception:
        pass
    print(f"[SECURITY] Generated new Fernet key at {key_file}.")
    return Fernet(new_key)


In [73]:

def encrypt_file(in_path: Path, out_path: Optional[Path] = None) -> Path:
    fernet = get_fernet()
    out_path = out_path or in_path.with_suffix(in_path.suffix + ".enc")
    data = in_path.read_bytes()
    enc = fernet.encrypt(data)
    out_path.write_bytes(enc)
    try:
        os.chmod(out_path, 0o600)
    except Exception:
        pass
    print(f"[SECURITY] Encrypted -> {out_path}")
    return out_path


In [74]:

def decrypt_file(enc_path: Path, out_path: Optional[Path] = None) -> Path:
    fernet = get_fernet()
    out_path = out_path or enc_path.with_name(enc_path.stem + ".decrypted.csv")
    enc = enc_path.read_bytes()
    dec = fernet.decrypt(enc)
    out_path.write_bytes(dec)
    print(f"[SECURITY] Decrypted -> {out_path}")
    return out_path


In [75]:
from pathlib import Path

output_dir = Path("scraped_books")
output_dir.mkdir(parents=True, exist_ok=True)

def main():
    print("[INFO] Starting scrape")
    df = scrape_books(max_pages=MAX_PAGES)

    if df.empty:
        print("[WARN] No data scraped. Exiting.")
        return

    csv_path = output_dir / "books_scraped.csv"
    df.to_csv(csv_path, index=False)
    print(f"[INFO] Saved temporary CSV -> {csv_path.resolve()}")

    enc_path = encrypt_file(csv_path, output_dir / "books_scraped.csv.enc")
    print(f"[SECURITY] Encrypted file saved -> {enc_path.resolve()}")

    try:
        csv_path.unlink()
        print(f"[SECURITY] Removed plaintext CSV -> {csv_path.name}")
    except Exception as e:
        print(f"[WARN] Could not remove plaintext CSV: {e}")

    print("[DONE] Encrypted file stored")

In [76]:
main()

[INFO] Starting scrape
[INFO] Fetching list page: https://books.toscrape.com/index.html
[INFO] Found 20 items on this page.
  [INFO] Fetching detail: https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
  [INFO] Fetching detail: https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html
  [INFO] Fetching detail: https://books.toscrape.com/catalogue/soumission_998/index.html
  [INFO] Fetching detail: https://books.toscrape.com/catalogue/sharp-objects_997/index.html
  [INFO] Fetching detail: https://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html
  [INFO] Fetching detail: https://books.toscrape.com/catalogue/the-requiem-red_995/index.html
  [INFO] Fetching detail: https://books.toscrape.com/catalogue/the-dirty-little-secrets-of-getting-your-dream-job_994/index.html
  [INFO] Fetching detail: https://books.toscrape.com/catalogue/the-coming-woman-a-novel-based-on-the-life-of-the-infamous-feminist-victoria-woodhull_993/index.h

In [77]:
decrypt_file(ENC_PATH, output_dir / "books_scraped.decrypted.csv")

[SECURITY] Decrypted -> scraped_books\books_scraped.decrypted.csv


WindowsPath('scraped_books/books_scraped.decrypted.csv')